<a href="https://colab.research.google.com/github/broskie2314/student-csc173-repository/blob/main/HateSpeechDetection_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
# Mount Google Drive
drive.mount('/content/drive')

# Project Directory
PROJECT_DIR = "/content/drive/MyDrive/HateSpeechDetection"

# Folder Structure
folders = [
    "dataset",
    "models"
]
for folder in folders:
  os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("Project Directory created!")

In [ ]:

import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

!git clone https://github.com/imperialite/filipino-tiktok-hatespeech.git

data_dir = "filipino-tiktok-hatespeech"
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
if not csv_files and os.path.exists(os.path.join(data_dir, "data")):
    csv_files = [os.path.join("data", f) for f in os.listdir(os.path.join(data_dir, "data")) if f.endswith('.csv')]
    raw_path = os.path.join(data_dir, csv_files[0])
else:
    raw_path = os.path.join(data_dir, csv_files[0])

print(f"Loading data from: {raw_path}")
df = pd.read_csv(raw_path)

# Standardize column names to 'text' and 'label'
text_col = [col for col in df.columns if 'text' in col.lower() or 'transcription' in col.lower()][0]
label_col = [col for col in df.columns if 'label' in col.lower() or 'vote' in col.lower()][0]

df = df[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).dropna()

# Text cleaning
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', str(text))  # Remove URLs
    return re.sub(r'\s+', ' ', text).strip()         # Normalize spaces

df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 0]
df['label'] = df['label'].astype(int)

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)


DATASET_PATH = os.path.join(PROJECT_DIR, "dataset")

train_df.to_csv(os.path.join(DATASET_PATH, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATASET_PATH, "val.csv"), index=False)
test_df.to_csv(os.path.join(DATASET_PATH, "test.csv"), index=False)

print("\n--- STEP 1 SUCCESSFUL ---")
print(f"Total samples:  {len(df)}")
print(f"Train samples:  {len(train_df)}")
print(f"Val samples:    {len(val_df)}")
print(f"Test samples:   {len(test_df)}")

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             roc_auc_score, confusion_matrix,
                             precision_score, recall_score)

tf.keras.utils.set_random_seed(42)  # reproducible results

# Load train and validation sets only (test set is kept untouched)
PROJECT_DIR = "/content/drive/MyDrive/HateSpeechDetection"
DATASET_PATH = os.path.join(PROJECT_DIR, "dataset")

train_df = pd.read_csv(os.path.join(DATASET_PATH, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATASET_PATH, "val.csv"))
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")

# Tokenization and padding
MAX_VOCAB = 10000
MAX_LEN = 64

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_df['text']), maxlen=MAX_LEN, padding='post')
X_val   = pad_sequences(tokenizer.texts_to_sequences(val_df['text']), maxlen=MAX_LEN, padding='post')

y_train = train_df['label'].values
y_val   = val_df['label'].values

# Build BiLSTM model
bilstm_model = models.Sequential([
    layers.Embedding(input_dim=MAX_VOCAB, output_dim=128),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

bilstm_model.compile(loss='binary_crossentropy',
                     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                     metrics=['accuracy'])

# Training with early stopping
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

start_train = time.time()
history = bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)
bilstm_train_time = time.time() - start_train

# Save model
MODEL_SAVE_PATH = os.path.join(PROJECT_DIR, "models", "bilstm_baseline.keras")
bilstm_model.save(MODEL_SAVE_PATH)

# ---- Evaluate on the VALIDATION set ----
start_infer = time.time()
y_pred_probs = bilstm_model.predict(X_val, verbose=0)
bilstm_infer_latency = ((time.time() - start_infer) / len(val_df)) * 1000  # ms per comment

y_pred = (y_pred_probs >= 0.5).astype(int).flatten()
y_pred_probs = y_pred_probs.flatten()

val_acc      = accuracy_score(y_val, y_pred)
val_macro_f1 = f1_score(y_val, y_pred, average='macro')
val_prec     = precision_score(y_val, y_pred)
val_rec      = recall_score(y_val, y_pred)
val_roc_auc  = roc_auc_score(y_val, y_pred_probs)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
val_fpr = fp / (fp + tn)

model_size_mb = os.path.getsize(MODEL_SAVE_PATH) / 1e6
n_params = bilstm_model.count_params()

print("=" * 60)
print("        BASELINE BiLSTM VALIDATION RESULTS")
print("=" * 60)
print(f"Macro F1-Score:             {val_macro_f1:.4f}")
print(f"Accuracy:                   {val_acc * 100:.2f}%")
print(f"Precision (Hate Speech):    {val_prec:.4f}")
print(f"Recall (Hate Speech):       {val_rec:.4f}")
print(f"ROC-AUC:                    {val_roc_auc:.4f}")
print(f"False Positive Rate (FPR):  {val_fpr * 100:.2f}%")
print(f"Confusion matrix:           TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Training Time:              {bilstm_train_time:.2f} seconds")
print(f"Inference Latency:          {bilstm_infer_latency:.2f} ms/comment")
print(f"Model Size:                 {model_size_mb:.1f} MB ({n_params:,} parameters)")
print("=" * 60)

print("\nDetailed Classification Report:")
print(classification_report(y_val, y_pred, target_names=["Safe (0)", "Hate Speech (1)"]))

# Save validation predictions for error analysis
val_df_results = val_df.copy()
val_df_results['bilstm_pred'] = y_pred
val_df_results['bilstm_prob'] = y_pred_probs
val_df_results.to_csv(os.path.join(DATASET_PATH, "bilstm_val_results.csv"), index=False)
print("Saved predictions to 'bilstm_val_results.csv' for error analysis.")

# Examples of mistakes (for the discussion)
errors = val_df_results[val_df_results['label'] != val_df_results['bilstm_pred']]
print("\nFalse negatives (hate missed):")
print(errors[errors['label'] == 1][['text', 'bilstm_prob']].sample(5, random_state=1).to_string())
print("\nFalse positives (safe flagged):")
print(errors[errors['label'] == 0][['text', 'bilstm_prob']].sample(5, random_state=1).to_string())

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import os
import time
import math
import numpy as np
import pandas as pd
import torch
import datasets
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             roc_auc_score, confusion_matrix,
                             precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, TrainerCallback, DataCollatorWithPadding, EarlyStoppingCallback)
from transformers.trainer_callback import ProgressCallback, PrinterCallback
from transformers.utils import logging

logging.set_verbosity_error()

# Load train and validation sets only (test set is kept untouched)
PROJECT_DIR = "/content/drive/MyDrive/HateSpeechDetection"
DATASET_PATH = os.path.join(PROJECT_DIR, "dataset")

train_df = pd.read_csv(os.path.join(DATASET_PATH, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATASET_PATH, "val.csv"))

print("Loaded datasets from Drive:")
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "jcblaise/roberta-tagalog-base"
MAX_LEN = 64
EPOCHS = 10
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(d):
    return Dataset.from_pandas(d[["text", "label"]].dropna().astype({"text": str, "label": int}).reset_index(drop=True))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = to_hf(train_df).map(tokenize, batched=True)
val_ds   = to_hf(val_df).map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Build the RoBERTa classifier
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "NOT_HATE_SPEECH", 1: "HATE_SPEECH"},
    label2id={"NOT_HATE_SPEECH": 0, "HATE_SPEECH": 1})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

class TrainLog(TrainerCallback):
    def __init__(self):
        self.train_loss = float("nan")
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_loss = logs["loss"]
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        m = metrics or {}
        print(f"Epoch {int(round(state.epoch))}/{EPOCHS} - loss: {self.train_loss:.4f} - "
              f"val_loss: {m.get('eval_loss', float('nan')):.4f} - "
              f"val_accuracy: {m.get('eval_accuracy', float('nan')):.4f} - "
              f"val_macro_f1: {m.get('eval_macro_f1', float('nan')):.4f}")

# Tuning & Training
steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)

args = TrainingArguments(
    output_dir="./roberta-tl-hatespeech",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=int(0.1 * steps_per_epoch * EPOCHS),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    disable_tqdm=True,
    report_to="none",
    seed=42,  # fixed seed so results are reproducible
)

trainer = Trainer(
    model=roberta_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3), TrainLog()],
)

for cb in (ProgressCallback, PrinterCallback):
    try:
        trainer.remove_callback(cb)
    except Exception:
        pass

# Measure training duration
print("\n--- TRAINING RoBERTa-Tagalog-base ---")
start_train = time.time()
trainer.train()
roberta_train_time = time.time() - start_train
print("Stopped at epoch:", round(trainer.state.epoch), "| Best checkpoint:", trainer.state.best_model_checkpoint)

# Save the trained model to Google Drive
MODEL_SAVE_PATH = os.path.join(PROJECT_DIR, "models", "roberta_tagalog_hatespeech")
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"Model saved to: {MODEL_SAVE_PATH}")

# ---- Evaluate on the VALIDATION set ----
y_val = np.array(val_ds["label"])

start_infer = time.time()
pred_out = trainer.predict(val_ds)
roberta_infer_latency = ((time.time() - start_infer) / len(val_df)) * 1000  # ms per comment

y_pred_probs = torch.softmax(torch.tensor(pred_out.predictions), dim=1)[:, 1].numpy()
y_pred = (y_pred_probs >= 0.5).astype(int).flatten()

val_acc      = accuracy_score(y_val, y_pred)
val_macro_f1 = f1_score(y_val, y_pred, average='macro')
val_prec     = precision_score(y_val, y_pred)
val_rec      = recall_score(y_val, y_pred)
val_roc_auc  = roc_auc_score(y_val, y_pred_probs)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
val_fpr = fp / (fp + tn)

model_size_mb = sum(
    os.path.getsize(os.path.join(MODEL_SAVE_PATH, f))
    for f in os.listdir(MODEL_SAVE_PATH) if f.endswith(".safetensors")
) / 1e6
n_params = sum(p.numel() for p in roberta_model.parameters())

print("=" * 60)
print("     RoBERTa-Tagalog-base VALIDATION RESULTS")
print("=" * 60)
print(f"Macro F1-Score:             {val_macro_f1:.4f}")
print(f"Accuracy:                   {val_acc * 100:.2f}%")
print(f"Precision (Hate Speech):    {val_prec:.4f}")
print(f"Recall (Hate Speech):       {val_rec:.4f}")
print(f"ROC-AUC:                    {val_roc_auc:.4f}")
print(f"False Positive Rate (FPR):  {val_fpr * 100:.2f}%")
print(f"Confusion matrix:           TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Training Time:              {roberta_train_time:.2f} seconds")
print(f"Inference Latency:          {roberta_infer_latency:.2f} ms/comment")
print(f"Model Size:                 {model_size_mb:.0f} MB ({n_params:,} parameters)")
print("=" * 60)

print("\nDetailed Classification Report:")
print(classification_report(y_val, y_pred, target_names=["Safe (0)", "Hate Speech (1)"]))

# Save validation predictions for error analysis
val_df_results = val_df.copy()
val_df_results['roberta_pred'] = y_pred
val_df_results['roberta_prob'] = y_pred_probs
val_df_results.to_csv(os.path.join(DATASET_PATH, "roberta_val_results.csv"), index=False)
print("Saved predictions to 'roberta_val_results.csv' for error analysis.")

# Examples of mistakes (for the discussion)
errors = val_df_results[val_df_results['label'] != val_df_results['roberta_pred']]
print("\nFalse negatives (hate missed):")
print(errors[errors['label'] == 1][['text', 'roberta_prob']].sample(5, random_state=1).to_string())
print("\nFalse positives (safe flagged):")
print(errors[errors['label'] == 0][['text', 'roberta_prob']].sample(5, random_state=1).to_string())